# Academy of Bards: parsed sample for review

Notebook version of `fanfic-sample-academy`. Parses the author index and the 5 sample author pages so you can check the output before running a full crawl of about 920 author pages.

**Network:** none if `verify_academy.ipynb` (or `fanfic-verify-academy`) has already saved the pages. Otherwise the missing pages are downloaded first (up to 6 requests, plus robots.txt).

Author names come from the index, not the author page, so every alias listed for a page (for example "Advocate (aka Blayne Cooper)" and "Cooper, Blayne (aka Advocate)") is kept.

In [ ]:
import os
import pathlib
import sys

# Find the repository root (the folder holding pyproject.toml) whether this
# notebook was opened from notebooks/ or from the root, and work from there,
# so saved pages and outputs always go to <repo>/data/ (or FANFIC_DATA_DIR).
REPO_ROOT = pathlib.Path.cwd().resolve()
while not (REPO_ROOT / "pyproject.toml").exists():
    if REPO_ROOT.parent == REPO_ROOT:
        raise RuntimeError("Open this notebook from inside the repository folder.")
    REPO_ROOT = REPO_ROOT.parent
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))  # works without `pip install -e .`
REPO_ROOT

## Settings

`CONTACT_EMAIL` is sent with every request (in the User-Agent) so site operators can reach whoever is crawling. Use your own address. Leave it blank to be asked the first time a download is needed, or set the `FANFIC_CONTACT_EMAIL` environment variable instead.

Only used if the pages have not been saved yet.

In [ ]:
CONTACT_EMAIL = ""  # your email address

In [ ]:
import dataclasses
import json

from fanfic_scraper.discover.academy import discover_academy
from fanfic_scraper.fetch import ACADEMY, build_default_fetcher
from fanfic_scraper.parse.academy import last_updated_date, parse_author_page
from fanfic_scraper.scripts.run_academy_sample import (
    build_display_names_by_normalized_url, lookup_display_names,
)

fetcher = build_default_fetcher(ACADEMY, contact_email=CONTACT_EMAIL or None)
index_page, entries, sample_pages = discover_academy(fetcher)
by_url = build_display_names_by_normalized_url(entries)

records = []
for page in sample_pages:
    records.extend(parse_author_page(page.body, page.url, lookup_display_names(by_url, page.url)))

print(f"{len(entries)} unique author URLs in the index")
print(f"Parsed {len(records)} records from {len(sample_pages)} sample author page(s)")

## Per-author summary

The "Last Updated" date is what a later incremental re-crawl would compare against.

In [ ]:
for page in sample_pages:
    page_records = [r for r in records if r.author_page_url == page.url]
    print(f"{page.url}")
    print(f"    last updated: {last_updated_date(page.body)}   stories: {len(page_records)}")

## Sample records

First 10 parsed records, full JSON.

In [ ]:
print(json.dumps([dataclasses.asdict(r) for r in records[:10]], indent=2, ensure_ascii=False))

## Series, incomplete stories and ebooks

In [ ]:
in_series = [r for r in records if r.series]
incomplete = [r for r in records if r.status == "incomplete"]
ebooks = [r for r in records if r.ebook_available]
print(f"{len(in_series)} in a series, {len(incomplete)} incomplete, {len(ebooks)} with an ebook")
for r in in_series:
    print(f"    {r.series}: {r.title!r} ({len(r.parts)} parts)")

## Write output to disk

The same file `fanfic-sample-academy` writes.

In [ ]:
from fanfic_scraper.fetch import default_data_dir

out_path = default_data_dir() / "academy_sample_output.json"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump([dataclasses.asdict(r) for r in records], f, indent=2, ensure_ascii=False)
print(f"Wrote {len(records)} records to {out_path}")